# AI-Art Forensics_2: CLIP Embedding Extraction

## Imports & constants

In [1]:
import os, sys
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.notebook import tqdm
import torch

DATA_DIR      = "/kaggle/input/datasets/ravidussilva/real-ai-art/Real_AI_SD_LD_Dataset"
OUT_DIR       = "/kaggle/working"
manifest_path = f"{OUT_DIR}/dataset_manifest.csv"
embed_path    = f"{OUT_DIR}/embeddings.npy"
meta_path     = f"{OUT_DIR}/metadata.csv"

def build_df():
    records = []
    for split in ["train", "test"]:
        split_path = os.path.join(DATA_DIR, split)
        for folder in sorted(os.listdir(split_path)):
            folder_path = os.path.join(split_path, folder)
            if not os.path.isdir(folder_path):
                continue
            if folder.startswith("AI_SD_"):
                source, style, label = "standard_diffusion", folder.replace("AI_SD_", ""), 1
            elif folder.startswith("AI_LD_"):
                source, style, label = "latent_diffusion", folder.replace("AI_LD_", ""), 1
            else:
                source, style, label = "human", folder, 0
            for fname in os.listdir(folder_path):
                if fname.lower().endswith((".jpg", ".jpeg", ".png", ".webp")):
                    records.append({"path": os.path.join(folder_path, fname),
                                    "split": split, "source": source,
                                    "style": style, "label": label,
                                    "filename": fname})
    return pd.DataFrame(records)


if os.path.exists(manifest_path):
    df = pd.read_csv(manifest_path)
    print(f"Manifest loaded from CSV: {len(df):,} images")
else:
    
    df = build_df()
    df.to_csv(manifest_path, index=False)
    print(f"Manifest built and saved: {len(df):,} images")

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

Manifest built and saved: 185,015 images
PyTorch : 2.10.0+cu128
CUDA    : True
GPU     : Tesla T4


## Section 1 — Install and load CLIP

We use OpenAI's original CLIP package with the **ViT-B/32** model (Vision Transformer, 32×32 patches). This produces 512-dimensional embeddings. It's the standard baseline model — strong enough to give us excellent separation between classes while being fast to run.

In [2]:
!pip install git+https://github.com/openai/CLIP.git

  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-6fmfrmng
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-6fmfrmng
  Resolved https://github.com/openai/CLIP.git to commit d05afc436d78f1c48dc0dbf8e5980a9d471f35f6
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.4 MB/s eta 0:00:00
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369547 sha256=7aa5e07f524bfbf2feb855adcc22437576189f054554ffef1575a6768a9de8df
  Stored in directory: /tmp/pip-ephem-wheel-cache-l_ou9u7c/wheels/35/3e/df/3d24cbfb3b6a06f17a2bfd7d1138900d4365d9028aa8f6e92f
Successfully built clip


In [3]:
import clip


In [4]:

model, preprocess = clip.load("ViT-B/32", device=DEVICE)
model.eval()  
print(f"CLIP model  : ViT-B/32")
print(f"Embedding dim: 512")
print(f"Device      : {DEVICE}")
print(f"Parameters  : {sum(p.numel() for p in model.parameters()):,}")

100%|████████████████████████████████████████| 338M/338M [00:01<00:00, 270MiB/s]


CLIP model  : ViT-B/32
Embedding dim: 512
Device      : cuda
Parameters  : 151,277,313


## Section 2 — Test on a single image

Before running 185k images, verify the pipeline works on one image. We should get a (1, 512) tensor.

In [5]:
test_samples = {}
for src in ['human', 'latent_diffusion', 'standard_diffusion']:
    row = df[df['source'] == src].iloc[0]
    test_samples[src] = row['path']

test_path = test_samples['human']
img       = preprocess(Image.open(test_path).convert("RGB")).unsqueeze(0).to(DEVICE)

with torch.no_grad():
    embedding = model.encode_image(img)

print(f"Input image path : {test_path}")
print(f"Embedding shape  : {embedding.shape}")   
print(f"Embedding dtype  : {embedding.dtype}")
print(f"Embedding norm   : {embedding.norm().item():.4f}")
print("Single-image test passed.")

Input image path : /kaggle/input/datasets/ravidussilva/real-ai-art/Real_AI_SD_LD_Dataset/train/art_nouveau/georges-rochegrosse_defile-des-centurions-pour-salammbo.jpg
Embedding shape  : torch.Size([1, 512])
Embedding dtype  : torch.float16
Embedding norm   : 10.1406
Single-image test passed.


## Section 3 — Extract embeddings for all 185k images

We process images in **batches of 128** (fits comfortably in T4/P100 VRAM). For each batch:
1. Load and preprocess images using CLIP's own transform (resize to 224×224, normalize)
2. Run through CLIP's image encoder
3. L2-normalize the embeddings (standard practice for cosine similarity downstream)
4. Accumulate in a pre-allocated numpy array

Failed images (corrupt files) are replaced with a zero vector and flagged in the metadata.

In [6]:
from torch.utils.data import Dataset, DataLoader

class ArtDataset(Dataset):
    def __init__(self, paths, transform):
        self.paths     = paths
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        try:
            img = Image.open(self.paths[idx]).convert("RGB")
            return self.transform(img), 1  # 1 = loaded ok
        except Exception:
       
            blank = Image.new("RGB", (224, 224), color=0)
            return self.transform(blank), 0  # 0 = failed

In [7]:

if os.path.exists(embed_path) and os.path.exists(meta_path):
    print("Embeddings already exist at:")
    print(f"  {embed_path}")
    print(f"  {meta_path}")
    print("Delete them manually if you want to re-extract. Skipping.")
else:
    print("No existing embeddings found. Starting extraction...")
    print(f"Will process {len(df):,} images in batches of 128.")

No existing embeddings found. Starting extraction...
Will process 185,015 images in batches of 128.


In [8]:
%%time

BATCH_SIZE   = 128
EMBED_DIM    = 512
N            = len(df)

all_embeddings = np.zeros((N, EMBED_DIM), dtype=np.float32)
load_ok        = np.ones(N, dtype=bool)  

dataset = ArtDataset(df['path'].tolist(), preprocess)
loader  = DataLoader(
    dataset,
    batch_size  = BATCH_SIZE,
    num_workers = 2,       
    pin_memory  = True,
    shuffle     = False    
)

idx = 0
with torch.no_grad():
    for images, ok_flags in tqdm(loader, desc="Extracting CLIP embeddings"):
        images = images.to(DEVICE)

        
        embeds = model.encode_image(images)          

        
        embeds = embeds / embeds.norm(dim=-1, keepdim=True)

        
        batch_np = embeds.cpu().float().numpy()
        bsz      = batch_np.shape[0]
        all_embeddings[idx:idx + bsz] = batch_np
        load_ok[idx:idx + bsz]        = ok_flags.numpy().astype(bool)
        idx += bsz

print(f"Extraction complete.")
print(f"  Total processed : {idx:,}")
print(f"  Failed images   : {(~load_ok).sum()} (replaced with zero vector)")
print(f"  Embeddings shape: {all_embeddings.shape}")

Extracting CLIP embeddings:   0%|          | 0/1446 [00:00<?, ?it/s]

Extraction complete.
  Total processed : 185,015
  Failed images   : 0 (replaced with zero vector)
  Embeddings shape: (185015, 512)
CPU times: user 3min 46s, sys: 29.4 s, total: 4min 16s
Wall time: 23min 47s


## Section 4 — Save embeddings and metadata

In [9]:

np.save(embed_path, all_embeddings)
print(f"Embeddings saved → {embed_path}")
print(f"  Shape : {all_embeddings.shape}")
print(f"  Size  : {all_embeddings.nbytes / 1e6:.1f} MB")

meta_df = df.copy()
meta_df['load_ok']    = load_ok
meta_df['embed_idx']  = np.arange(N)   
meta_df.to_csv(meta_path, index=False)
print(f"Metadata saved  → {meta_path}")
print(f"  Columns: {list(meta_df.columns)}")

Embeddings saved → /kaggle/working/embeddings.npy
  Shape : (185015, 512)
  Size  : 378.9 MB
Metadata saved  → /kaggle/working/metadata.csv
  Columns: ['path', 'split', 'source', 'style', 'label', 'filename', 'load_ok', 'embed_idx']


## Section 5 — Sanity checks

Before moving to notebook 3, verify the embeddings are sensible:
1. Check embedding norms (should all be ~1.0 after L2 normalization)
2. Compute cosine similarity between a human image and an AI image — should be less than 1.0
3. Compute cosine similarity between two human images of the same style — should be higher than human vs AI

In [10]:

E    = np.load(embed_path)
meta = pd.read_csv(meta_path)

print(f"Reloaded embeddings shape : {E.shape}")
print(f"Reloaded metadata rows    : {len(meta):,}")

norms = np.linalg.norm(E, axis=1)
print(f"\nEmbedding norms:")
print(f"  Mean : {norms.mean():.6f}")
print(f"  Std  : {norms.std():.6f} ")
print(f"  Min  : {norms.min():.6f}")
print(f"  Max  : {norms.max():.6f}")

Reloaded embeddings shape : (185015, 512)
Reloaded metadata rows    : 185,015

Embedding norms:
  Mean : 1.000000
  Std  : 0.000240 
  Min  : 0.999419
  Max  : 1.000580


In [11]:
def cos_sim(a, b):
    return float(np.dot(a, b))
style_to_test = 'impressionism'

h_idx  = meta[(meta['source'] == 'human') & 
               (meta['style'] == style_to_test)].index[:10].tolist()
ld_idx = meta[(meta['source'] == 'latent_diffusion') & 
               (meta['style'] == style_to_test)].index[:10].tolist()
sd_idx = meta[(meta['source'] == 'standard_diffusion') & 
               (meta['style'] == style_to_test)].index[:10].tolist()

hh   = np.mean([cos_sim(E[h_idx[i]],  E[h_idx[j]])  for i in range(10) for j in range(10) if i != j])
hld  = np.mean([cos_sim(E[h_idx[i]],  E[ld_idx[j]]) for i in range(10) for j in range(10)])
hsd  = np.mean([cos_sim(E[h_idx[i]],  E[sd_idx[j]]) for i in range(10) for j in range(10)])
ldsd = np.mean([cos_sim(E[ld_idx[i]], E[sd_idx[j]]) for i in range(10) for j in range(10)])

print(f"Style-controlled cosine similarity ({style_to_test}):")
print(f"  Human vs Human             : {hh:.4f}")
print(f"  Human vs Latent Diffusion  : {hld:.4f}")
print(f"  Human vs Standard Diffusion: {hsd:.4f}")
print(f"  Latent Diff vs Std Diff    : {ldsd:.4f}")

Style-controlled cosine similarity (impressionism):
  Human vs Human             : 0.6776
  Human vs Latent Diffusion  : 0.6344
  Human vs Standard Diffusion: 0.6345
  Latent Diff vs Std Diff    : 0.7127


In [12]:

import matplotlib.pyplot as plt

print("Mean embedding per source (first 5 dims):")
for src in ['human', 'latent_diffusion', 'standard_diffusion']:
    idx  = meta[meta['source'] == src].index
    mean = E[idx].mean(axis=0)
    std  = E[idx].std(axis=0)
    print(f"  {src:25s}: mean={mean[:5].round(4)}, std={std[:5].round(4)}")

print()
print("=" * 50)
print("=" * 50)
print(f"embeddings.npy → {all_embeddings.shape} ({all_embeddings.nbytes/1e6:.0f} MB)")
print(f"metadata.csv   → {len(meta_df):,} rows")
print()

Mean embedding per source (first 5 dims):
  human                    : mean=[-0.004  -0.0011 -0.0012  0.0039  0.0216], std=[0.0296 0.024  0.0184 0.0223 0.0261]
  latent_diffusion         : mean=[ 0.0267  0.0023 -0.005   0.0028  0.0172], std=[0.0281 0.0219 0.0206 0.0205 0.0223]
  standard_diffusion       : mean=[ 0.0303 -0.0035  0.0029 -0.0036  0.0122], std=[0.0287 0.0222 0.0219 0.0187 0.0243]

embeddings.npy → (185015, 512) (379 MB)
metadata.csv   → 185,015 rows

